<a href="https://colab.research.google.com/github/matjesg/deepflash2/blob/colab-test-modernize/colab_test_modernize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# deepflash2 — modernized-stack Colab test

Tests the `feat/modernize-stack` branch (PR #70: upstream segmentation-models-pytorch, timm 1.0, torch 2.x, numpy 2, pandas 2/3) on Colab's current Python.

**How to run:** `Runtime` → `Run all`. If an import fails right after install (Colab preloads numpy/torch), do `Runtime` → `Restart session`, then run the cells again from the imports.

In [ ]:
#@title 1. Install deepflash2 from the modernization branch (~2-3 min)
!pip install -Uqq "git+https://github.com/matjesg/deepflash2.git@feat/modernize-stack"
from IPython.display import clear_output
import deepflash2, torch, numpy, pandas, segmentation_models_pytorch as smp, timm, albumentations
clear_output(wait=True)
print("deepflash2 :", deepflash2.__version__)
print("python     :", __import__('sys').version.split()[0])
print("torch      :", torch.__version__, "| cuda:", torch.cuda.is_available())
print("numpy      :", numpy.__version__)
print("pandas     :", pandas.__version__)
print("smp        :", smp.__version__)
print("timm       :", timm.__version__)
print("albumentat.:", albumentations.__version__)

In [ ]:
#@title 2. Smoke test: import everything + build models + forward pass
import torch, deepflash2.all
from deepflash2.models import create_smp_model
x = torch.randn(1, 1, 128, 128)
def check(arch, encoder_name):
    # eval() is required: DeepLabV3+ has a 1x1 global-pool feature that BatchNorm
    # can't handle in train mode with batch=1. Real training uses batch>1.
    m = create_smp_model(arch, encoder_name=encoder_name, encoder_weights=None, in_channels=1, classes=2).eval()
    with torch.no_grad():
        out = tuple(m(x).shape)
    print(f"{arch + ' / ' + encoder_name:34s} -> {out}")
    return out
for arch in ['Unet', 'UnetPlusPlus', 'FPN', 'DeepLabV3Plus', 'PAN', 'PSPNet', 'Linknet', 'DeepLabV3']:
    check(arch, 'resnet18')
# riskiest path: ConvNeXt-Unet (exercises the code where the old decoder monkey-patch was removed)
out = check('Unet', 'tu-convnext_tiny')
assert out == (1, 2, 128, 128), 'ConvNeXt-Unet output is not full resolution!'
print('\nSmoke test passed ✅  (all architectures build + forward at full resolution)')

In [ ]:
#@title 3. Launch the deepflash2 GUI (tests the ipywidgets fix from #51)
#@markdown After it renders, click **Load Sample Data** in the sidebar to try GT estimation / training / prediction.
from deepflash2.gui import GUI, _connect_to_drive
path = _connect_to_drive()
GUI(path)

In [ ]:
#@markdown On Colab: run this cell during training/prediction to avoid a disconnect. Stop execution to continue.
try:
    while True: pass
except:
    pass